# [재실행 전용 · 2026-10-04 회계 수정] crypto_lsre_caan_baseline_colab.ipynb

원본 `crypto_lsre_caan_baseline_colab.ipynb`에서 **이번 재실행에 필요한 셀만** 뽑은 노트북이다. 학습 셀(`.learn`)은 빠져 있고, 저장된 모델을 불러와 평가만 한다.
위에서부터 **순서대로 전부 실행**하면 된다. 결과 CSV는 원본과 같은 이름으로 `data/crypto/`에 저장된다(실행 전 백업 셀 먼저).

결과: 표 2 LSRE-CAAN 행.


In [ ]:
import shutil, glob, os
from google.colab import drive
drive.mount('/content/drive')
src = '/content/drive/MyDrive/졸업프로젝트/data/crypto'
dst = src + '/_pre_accounting_fix_20261004'
if not os.path.exists(dst):
    os.makedirs(dst)
    for p in glob.glob(src + '/*.csv') + glob.glob(src + '/*.png'):
        shutil.copy2(p, dst)
print('백업 폴더:', dst, len(os.listdir(dst)), '개')


# LSRE-CAAN 베이스라인 — 2020년대 딥러닝 포트폴리오 방법과의 비교 (`paper/RELATED_WORK.md` §5.5)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium pyarrow


## 1. 설정

In [ ]:
import warnings
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

warnings.filterwarnings('ignore')
plt.rcParams['axes.unicode_minus'] = False

DRIVE_ROOT = Path('/content/drive/MyDrive/졸업프로젝트')
DATA_DIR   = DRIVE_ROOT / 'data'
CRYPTO_DIR = DATA_DIR / 'crypto'
FEAT_DIR   = CRYPTO_DIR / 'features_5m'
MODEL_DIR  = DRIVE_ROOT / 'models'

TC         = 0.001
BUCKET_SEC = 1800

RL_TRAIN_START = '2025-01-01'
RL_TRAIN_END   = '2025-12-31'
RL_TEST_START  = '2026-01-01'
RL_TEST_END    = '2099-12-31'
VALID_START, VALID_END = '2025-10-01', '2025-12-31'

WINDOW       = 50    # EIIE 노트북과 동일(공정 비교)
N_CHANNELS   = 3     # [close, high, low] / 마지막 종가 비율 — EIIE 노트북과 동일

# LSRE-CAAN 하이퍼파라미터 — 공개 구현(jiahaoli57/LSRE-CAAN) 기본값 그대로 사용
LATENT_DIM      = 32
NUM_LATENTS     = 1
CROSS_HEADS     = 1
LATENT_HEADS    = 1
NUM_LAYERS      = 1

LSRE_CAAN_TIMESTEPS = 100_000   # β=-30(§13-C) · EIIE 노트북과 동일 예산

LSRE_CAAN_MODEL_DIR = MODEL_DIR / 'crypto_lsre_caan_baseline'
LSRE_CAAN_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('경로 확인 완료')


## 2. 데이터 로드 — 심볼 목록 + RetTarget_6b (임베딩 불필요, features_5m만 사용)

In [ ]:
files = sorted(glob.glob(str(FEAT_DIR / '*.parquet')))
assert files, f'피처 parquet 없음: {FEAT_DIR}'
# [2026-10-04] 2026-08-18 PIT walk-forward용으로 추가된 3종목(backfill_pit_extra.py)은 원 44종목 유니버스가 아님 — 제외
PIT_EXTRA = {'DOTUSDT', 'THETAUSDT', 'XTZUSDT'}
files = [fp for fp in files if Path(fp).stem not in PIT_EXTRA]
symbols = sorted(Path(fp).stem for fp in files)
N_ASSETS = len(symbols)
assert N_ASSETS == 44, f'유니버스가 44종목이 아님({N_ASSETS}) — 저장된 모델과 차원 불일치'
print(f'심볼 {N_ASSETS}개: {symbols[:5]} ...')

rl_start_ts = pd.Timestamp(RL_TRAIN_START, tz='UTC')
ret_index = {}
all_buckets_set = set()

for fp in files:
    sym = Path(fp).stem
    df = pd.read_parquet(fp, columns=['datetime', 'RetTarget_6b'])
    df = df[df['datetime'] >= rl_start_ts]
    epoch = df['datetime'].values.astype('int64') // 10**9
    y     = df['RetTarget_6b'].values
    b     = epoch // BUCKET_SEC
    for bb, r in zip(b, y):
        if not np.isnan(r):
            bb = int(bb)
            ret_index[(sym, bb)] = float(r)
            all_buckets_set.add(bb)

all_buckets = sorted(all_buckets_set)
print(f'수익률 인덱스: {len(ret_index):,}개 | 전체 30분 버킷: {len(all_buckets):,}개')


## 3. 버킷 분할

In [ ]:
def bucket_to_ts(b):
    return pd.Timestamp(b * BUCKET_SEC, unit='s', tz='UTC')

def bucket_to_date(b):
    return bucket_to_ts(b).strftime('%Y-%m-%d')

train_buckets = [b for b in all_buckets if RL_TRAIN_START <= bucket_to_date(b) <= RL_TRAIN_END]
valid_buckets = [b for b in train_buckets if VALID_START <= bucket_to_date(b) <= VALID_END]
test_buckets  = [b for b in all_buckets if RL_TEST_START  <= bucket_to_date(b) <= RL_TEST_END]

print(f'train: {bucket_to_ts(train_buckets[0])} ~ {bucket_to_ts(train_buckets[-1])} ({len(train_buckets):,}스텝)')
print(f'valid: {bucket_to_ts(valid_buckets[0])} ~ {bucket_to_ts(valid_buckets[-1])} ({len(valid_buckets):,}스텝)')
print(f'test : {bucket_to_ts(test_buckets[0])} ~ {bucket_to_ts(test_buckets[-1])} ({len(test_buckets):,}스텝)')


## 4. 원시 가격 행렬 구성 — close/high/low, 30분 리샘플 (EIIE 노트북과 동일)

In [ ]:
%%time
BUCKET_MIN = train_buckets[0] - WINDOW - 5   # 워밍업 여유
BUCKET_MAX = test_buckets[-1]
global_idx = np.arange(BUCKET_MIN, BUCKET_MAX + 1)
pos_of = {b: i for i, b in enumerate(global_idx)}

close_mat = np.full((N_ASSETS, len(global_idx)), np.nan, dtype=np.float32)
high_mat  = np.full_like(close_mat, np.nan)
low_mat   = np.full_like(close_mat, np.nan)

for i, sym in enumerate(symbols):
    df = pd.read_parquet(FEAT_DIR / f'{sym}.parquet', columns=['datetime', 'open', 'high', 'low', 'close'])
    epoch = df['datetime'].values.astype('int64') // 10**9
    b = epoch // BUCKET_SEC
    tmp = pd.DataFrame({'high': df['high'].values, 'low': df['low'].values, 'close': df['close'].values}, index=b)
    agg = tmp.groupby(level=0).agg({'high': 'max', 'low': 'min', 'close': 'last'})
    agg = agg.reindex(global_idx)
    agg = agg.ffill().bfill()
    close_mat[i] = agg['close'].to_numpy(dtype=np.float32)
    high_mat[i]  = agg['high'].to_numpy(dtype=np.float32)
    low_mat[i]   = agg['low'].to_numpy(dtype=np.float32)

assert not np.isnan(close_mat).any(), '가격 행렬에 NaN 남음 — 전 구간 데이터 없는 심볼 확인 필요'
print(f'가격 행렬: {close_mat.shape} (심볼 × 버킷)')


## 5. PriceWindowPortfolioEnv — EIIE 노트북과 완전히 동일한 환경(공정 비교)

In [ ]:
# [2026-10-04 회계 수정] 가격 변동 후 실제 보유 비중(drift)·거래 가능 종목 마스크.
# 이전 버전은 수익 반영 뒤에도 직전 '목표' 비중을 보유 비중으로 저장해, 가격 변동으로 틀어진 비중을
# 되돌리는 재조정 거래(등가중 포함)의 비용이 0으로 계산됐다.
def _drift_weights(w, rets):
    """수익 반영 뒤 실제 보유 비중 w_i(1+r_i)/(1+wᵀr). 합<1이면 나머지는 현금(수익률 0)으로 남는다."""
    w = np.asarray(w, dtype=np.float64)
    g = 1.0 + float(np.dot(w, rets))
    if g <= 1e-12:
        return w.astype(np.float32)
    return (w * (1.0 + np.asarray(rets, dtype=np.float64)) / g).astype(np.float32)


def _mask_untradable(w, tradable, renorm=True):
    """수익률 데이터가 없는(상장 전·상폐 후) 종목 비중을 0으로. renorm=False면 빠진 몫은 현금으로 남는다."""
    w = np.where(tradable, np.asarray(w, dtype=np.float64), 0.0)
    s = w.sum()
    return (w / s if renorm and s > 1e-12 else w).astype(np.float32)


class PriceWindowPortfolioEnv(gym.Env):
    '''crypto_eiie_baseline_colab.ipynb의 EIIEPortfolioEnv와 동일 로직 — 원시 가격 윈도우 +
    직전 비중, (N+1)-dim 행동(자산 N개 + 현금 1개) → softmax. 거래비용은 자산 레그에만 부과.
    이 노트북에서는 네트워크(§6)만 EIIE와 다르고 환경은 완전히 동일하게 재사용해, 두 베이스라인
    비교가 아키텍처 차이만을 반영하도록 한다.'''
    metadata = {'render_modes': []}

    def __init__(self, buckets, symbols, close_mat, high_mat, low_mat, pos_of,
                 window=WINDOW, tc=TC):
        super().__init__()
        self.buckets   = buckets
        self.symbols   = symbols
        self.n_assets  = len(symbols)
        self.close_mat = close_mat
        self.high_mat  = high_mat
        self.low_mat   = low_mat
        self.pos_of    = pos_of
        self.window    = window
        self.tc        = tc

        obs_dim = self.n_assets * N_CHANNELS * window + self.n_assets
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(obs_dim,), dtype=np.float32)
        self.action_space      = spaces.Box(-3.0, 3.0, shape=(self.n_assets + 1,), dtype=np.float32)

    def _price_window(self, bucket):
        pos = self.pos_of[bucket]
        sl = slice(pos - self.window + 1, pos + 1)
        c = self.close_mat[:, sl]
        h = self.high_mat[:, sl]
        l = self.low_mat[:, sl]
        last_close = c[:, -1:]
        last_close = np.where(last_close == 0, 1e-8, last_close)
        rc = (c / last_close).astype(np.float32)
        rh = (h / last_close).astype(np.float32)
        rl = (l / last_close).astype(np.float32)
        return np.stack([rc, rh, rl], axis=1)   # (n_assets, 3, window)

    def _asset_ret(self, bucket):
        rets = np.array([ret_index.get((sym, bucket), 0.0) for sym in self.symbols], dtype=np.float32)
        return np.concatenate([rets, [0.0]])

    def _tradable(self, bucket):
        """거래 가능 마스크(마지막 = 현금, 항상 가능)"""
        return np.array([(sym, bucket) in ret_index for sym in self.symbols] + [True])

    def _get_obs(self):
        b = self.buckets[self.t]
        pw = self._price_window(b)
        return np.concatenate([pw.flatten(), self.weights[:-1]]).astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.t               = 0
        self.weights         = np.zeros(self.n_assets + 1, dtype=np.float32)
        self.weights[-1]     = 1.0
        self.portfolio_value = 1.0
        self.value_history   = [1.0]
        self.turnover_history = []
        return self._get_obs(), {}

    def step(self, action):
        action = np.exp(action - action.max())
        w = (action / action.sum()).astype(np.float32)
        w = _mask_untradable(w, self._tradable(self.buckets[self.t]))  # 거래 불가 종목 제외

        turnover = float(np.abs(w[:-1] - self.weights[:-1]).sum())
        tc_cost  = self.tc * turnover

        b = self.buckets[self.t]
        asset_rets = self._asset_ret(b)
        port_ret = float((w * asset_rets).sum())

        self.portfolio_value *= (1 + port_ret - tc_cost)
        self.weights = _drift_weights(w, asset_rets)  # 가격 변동 후 실제 보유 비중
        self.value_history.append(self.portfolio_value)
        self.turnover_history.append(turnover)
        self.t += 1

        done = (self.t >= len(self.buckets) - 1)
        reward = float(port_ret - tc_cost)
        info = {'portfolio_value': self.portfolio_value, 'port_ret': port_ret, 'turnover': turnover}
        return self._get_obs(), reward, done, False, info


# 스모크 테스트
_env = PriceWindowPortfolioEnv(train_buckets[:200], symbols, close_mat, high_mat, low_mat, pos_of)
check_env(_env, warn=True)
obs, _ = _env.reset()
a = np.random.randn(_env.n_assets + 1).astype(np.float32)
obs, r, done, _, info = _env.step(a)
print(f'obs shape: {obs.shape} | action shape: {_env.action_space.shape} | reward: {r:.5f}')


## 6. LSRE-CAAN 네트워크 (Li, Zhang, Yang, & Chen, 2023 이식)

In [ ]:
class LSRE(nn.Module):
    """Long Sequence Representations Extractor — Perceiver류 attention bottleneck.
    소수(NUM_LATENTS)의 학습 가능한 latent가 가격 윈도우 시퀀스에 cross-attention으로 접근해
    압축한 뒤, latent끼리 self-attention으로 정제한다. 원 Transformer의 시퀀스길이 제곱 복잡도를
    latent 수 기준 선형으로 낮추는 것이 이 모듈의 핵심 동기(원 논문 근거)."""

    def __init__(self, input_dim, window, latent_dim=LATENT_DIM, num_latents=NUM_LATENTS,
                 cross_heads=CROSS_HEADS, latent_heads=LATENT_HEADS, num_layers=NUM_LAYERS):
        super().__init__()
        self.pos_emb    = nn.Embedding(window, input_dim)
        self.input_proj = nn.Linear(input_dim, latent_dim)
        self.latents    = nn.Parameter(torch.randn(num_latents, latent_dim) * 0.02)
        self.cross_attn = nn.MultiheadAttention(latent_dim, cross_heads, batch_first=True)
        self.cross_ln   = nn.LayerNorm(latent_dim)
        self.self_layers = nn.ModuleList([
            nn.ModuleDict({
                'attn': nn.MultiheadAttention(latent_dim, latent_heads, batch_first=True),
                'ln1':  nn.LayerNorm(latent_dim),
                'ff':   nn.Sequential(nn.Linear(latent_dim, latent_dim * 2), nn.GELU(),
                                       nn.Linear(latent_dim * 2, latent_dim)),
                'ln2':  nn.LayerNorm(latent_dim),
            }) for _ in range(num_layers)
        ])

    def forward(self, x):
        # x: (B, window, input_dim)
        B, W, _ = x.shape
        pos = self.pos_emb(torch.arange(W, device=x.device)).unsqueeze(0)
        x = self.input_proj(x + pos)                              # (B, window, latent_dim)
        latents = self.latents.unsqueeze(0).expand(B, -1, -1)     # (B, num_latents, latent_dim)
        attn_out, _ = self.cross_attn(latents, x, x)
        latents = self.cross_ln(latents + attn_out)
        for layer in self.self_layers:
            attn_out, _ = layer['attn'](latents, latents, latents)
            latents = layer['ln1'](latents + attn_out)
            latents = layer['ln2'](latents + layer['ff'](latents))
        return latents[:, -1, :]                                   # (B, latent_dim) — 마지막 latent


class CAAN(nn.Module):
    """Cross-Asset Attention Network — LSRE가 만든 자산별 latent 표현들 사이에 cross-attention을
    적용해 자산 간 상호작용을 반영한 최종 종목별 점수를 산출(원 논문 표기와 세부 구현은 다를 수
    있음 — 공개 구현에서 확인 가능한 수준의 메커니즘만 이식)."""

    def __init__(self, latent_dim=LATENT_DIM, heads=1):
        super().__init__()
        self.attn        = nn.MultiheadAttention(latent_dim, heads, batch_first=True)
        self.ln          = nn.LayerNorm(latent_dim)
        self.score_head  = nn.Linear(latent_dim, 1)

    def forward(self, asset_latents):
        # asset_latents: (B, n_assets, latent_dim)
        attn_out, _ = self.attn(asset_latents, asset_latents, asset_latents)
        x = self.ln(asset_latents + attn_out)
        return self.score_head(x).squeeze(-1)   # (B, n_assets)


class LSRECAANNet(nn.Module):
    def __init__(self, n_assets, window, n_channels):
        super().__init__()
        self.n_assets, self.window, self.n_channels = n_assets, window, n_channels
        self.lsre = LSRE(n_channels, window)
        self.caan = CAAN(LATENT_DIM)
        self.cash_bias = nn.Parameter(torch.zeros(1))   # EIIE와 동일한 현금 편향 관례(원 논문에는 없음, 우리 환경의 현금 레그 대응)

    def forward(self, price_window):
        # price_window: (B, n_assets, n_channels, window)
        B = price_window.shape[0]
        x = price_window.permute(0, 1, 3, 2).reshape(B * self.n_assets, self.window, self.n_channels)
        asset_repr = self.lsre(x).reshape(B, self.n_assets, -1)   # (B, n_assets, latent_dim)
        scores = self.caan(asset_repr)                             # (B, n_assets)
        cash = self.cash_bias.expand(B, 1)
        return torch.cat([scores, cash], dim=-1)                   # (B, n_assets+1)


class LSRECAANFeaturesExtractor(BaseFeaturesExtractor):
    """SB3 정책망에 꽂는 래퍼 — 관측(flatten)을 (가격윈도우, 직전비중)으로 분리하되, 이 네트워크는
    직전비중(PVM)을 쓰지 않는다(§0 고지 참고) — EIIEFeaturesExtractor와 동일한 관측 형식을 공유해
    같은 PriceWindowPortfolioEnv를 그대로 재사용할 수 있게만 한다."""

    def __init__(self, observation_space, n_assets=N_ASSETS, window=WINDOW, n_channels=N_CHANNELS):
        features_dim = n_assets + 1
        super().__init__(observation_space, features_dim=features_dim)
        self.n_assets, self.window, self.n_channels = n_assets, window, n_channels
        self.net = LSRECAANNet(n_assets, window, n_channels)
        self._pw_size = n_assets * n_channels * window

    def forward(self, observations):
        pw_flat = observations[:, :self._pw_size]
        pw = pw_flat.reshape(-1, self.n_assets, self.n_channels, self.window)
        return self.net(pw)


policy_kwargs = dict(
    features_extractor_class=LSRECAANFeaturesExtractor,
    features_extractor_kwargs=dict(n_assets=N_ASSETS, window=WINDOW, n_channels=N_CHANNELS),
    net_arch=[],   # feature extractor 출력(자산별 점수+현금 편향)을 거의 그대로 정책에 사용
)
print('LSRE-CAAN 네트워크 정의 완료')


## 7. 성과 계산 함수

In [ ]:
def compute_perf(values, steps_per_year=48 * 365):
    rets       = np.diff(values) / values[:-1]
    total_ret  = values[-1] / values[0] - 1
    annual_ret = (1 + total_ret) ** (steps_per_year / len(rets)) - 1
    sharpe     = rets.mean() / (rets.std() + 1e-8) * np.sqrt(steps_per_year)
    max_dd     = ((values / np.maximum.accumulate(values)) - 1).min()
    return {'총 수익률': total_ret, '연환산 수익률': annual_ret, 'Sharpe': sharpe, 'MDD': max_dd}


def perf_row(values, turnovers, tc=TC):
    perf = compute_perf(values)
    t = np.asarray(turnovers, dtype=np.float64)
    perf['평균 턴오버/스텝'] = t.mean()
    perf['누적 비용 드래그'] = 1 - np.prod(1 - tc * t)
    return perf


def equal_weight_backtest(buckets):
    env = PriceWindowPortfolioEnv(buckets, symbols, close_mat, high_mat, low_mat, pos_of)
    obs, _ = env.reset()
    done = False
    action = np.zeros(env.n_assets + 1, dtype=np.float32)
    action[:-1] = 1.0
    action[-1] = -10.0
    while not done:
        obs, _, done, _, info = env.step(action)
    return np.array(env.value_history), list(env.turnover_history)


## 8. LSRE-CAAN 학습 (PPO, EIIE와 동일 예산 100k 스텝)

In [ ]:
# 재실행용 로더 (idempotent)
if 'lsre_caan_model' not in globals():
    lsre_caan_model = PPO.load(str(LSRE_CAAN_MODEL_DIR / 'lsre_caan_seed42.zip'),
                                custom_objects={'policy_kwargs': policy_kwargs})
    print('LSRE-CAAN 모델 로드 완료')


## 9. 평가 — valid/test, 기존 결과와 비교

In [ ]:
def run_model(model, buckets):
    env = PriceWindowPortfolioEnv(buckets, symbols, close_mat, high_mat, low_mat, pos_of)
    obs, _ = env.reset()
    done = False
    while not done:
        a, _ = model.predict(obs, deterministic=True)
        obs, _, done, _, info = env.step(a)
    return np.array(env.value_history), list(env.turnover_history)


print('=== LSRE-CAAN 평가 — valid vs test ===\n')

ew_valid_vals, ew_valid_turns = equal_weight_backtest(valid_buckets)
ew_test_vals,  ew_test_turns  = equal_weight_backtest(test_buckets)

lc_valid_vals, lc_valid_turns = run_model(lsre_caan_model, valid_buckets)
lc_test_vals,  lc_test_turns  = run_model(lsre_caan_model, test_buckets)

comparison_valid = {
    '등가중 벤치마크(이 노트북 재계산)': perf_row(ew_valid_vals, ew_valid_turns),
    'LSRE-CAAN(PPO 학습, seed=42)':     perf_row(lc_valid_vals, lc_valid_turns),
    # 참고용 — 다른 노트북(crypto_eiie_baseline_colab.ipynb)에서 이미 확인된 값, 이 노트북에서 재계산 아님
    'EIIE(참고치)':                     {'총 수익률': float('nan'), 'Sharpe': -0.969, 'MDD': float('nan')},
}
comparison_test = {
    '등가중 벤치마크(이 노트북 재계산)': perf_row(ew_test_vals, ew_test_turns),
    'LSRE-CAAN(PPO 학습, seed=42)':     perf_row(lc_test_vals, lc_test_turns),
    # 참고용 — 다른 노트북 결과, 재계산 아님(버킷·TC는 동일하게 맞췄으나 완전한 재현은 아님)
    'EIIE(참고치)':                     {'총 수익률': 0.0018, 'Sharpe': 0.275, 'MDD': float('nan')},
    'β=-30 RL(배포, 참고치)':           {'총 수익률': 0.0566, 'Sharpe': 1.1556, 'MDD': -0.0743},
    '모멘텀 top-10(참고치)':            {'총 수익률': 1.0604, 'Sharpe': 2.4433, 'MDD': -0.2796},
}

valid_df = pd.DataFrame(comparison_valid).T
test_df  = pd.DataFrame(comparison_test).T

print('--- valid(2025 Q4) ---')
print(valid_df.to_string(float_format=lambda x: f'{x:.4f}'))
print('\n--- test(2026~) ---')
print(test_df.to_string(float_format=lambda x: f'{x:.4f}'))

valid_df.to_csv(CRYPTO_DIR / 'crypto_lsre_caan_valid.csv', encoding='utf-8-sig')
test_df.to_csv(CRYPTO_DIR / 'crypto_lsre_caan_test.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_lsre_caan_valid.csv, crypto_lsre_caan_test.csv')
print('\n(주의: EIIE·β=-30·모멘텀 참고치 행은 이 노트북에서 재계산한 게 아니라 다른 노트북 결과를')
print(' 그대로 옮겨온 것 — 버킷·TC는 동일하게 맞췄으나 완전한 재현은 아님. 엄밀 비교는 등가중·')
print(' LSRE-CAAN 두 행 사이에서만 유효.)')


## 10. 판정 기준